# Day 1-03｜Roboflow BBOX Dataset 與 YOLO Detection 訓練準備
> Python 籃球運動資料分析課程  
> 本單元定義籃球偵測類別、Roboflow YOLO export 放置位置，並用已訓練 YOLO detector 檢查實際影片 frame。  
> 修課背景：具備基礎 Python 語法即可；不預設電腦視覺或運動資料分析經驗。

## 學習目標
- 建立 detection dataset 的類別規格與資料夾結構。
- 檢查 Roboflow YOLO detection export 是否可被 Ultralytics 讀取。
- 使用教師提供的 YOLO detector 權重產生實際偵測結果。

## 資料放置位置
- Roboflow YOLO detection export：`assets/datasets/roboflow_bbox_yolo/`
- 已訓練 detector 權重：`assets/models/detectors/yolo26n_basketball_player_best.pt`


## 課程流程
1. 確認籃球偵測類別與 Roboflow export 結構。
2. 若是學生自己的 Roboflow 專案，先在網頁完成 bbox 標註。
3. 到 Roboflow 專案的 `Versions` 頁按 `Generate New Version` / `Publish`。
4. 把新的 `workspace slug`、`project slug`、`version number` 填回本 notebook。
5. 檢查 `assets/datasets/roboflow_bbox_yolo/`。
6. 閱讀實際訓練程式；預設不重新訓練。
7. 使用已訓練 detector 對參考影片 frame 執行推論。


In [ ]:
from pathlib import Path
import sys

try:
    from google.colab import drive  # type: ignore[import-not-found]

    drive.mount("/content/drive")
except ModuleNotFoundError:
    pass

bootstrap_candidates = [
    Path("/content/drive/MyDrive/basketball_hackathon/course"),
    Path.cwd().resolve(),
    *Path.cwd().resolve().parents,
]
for candidate in bootstrap_candidates:
    if (candidate / "src" / "course_setup.py").exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break
else:
    raise FileNotFoundError(
        "找不到 src/course_setup.py。請先執行 init_colab.ipynb，確認課程已同步到 /content/drive/MyDrive/basketball_hackathon/course。"
    )

from src.course_setup import DEFAULT_COURSE_ROOT, bootstrap_course_notebook  # noqa: E402

COURSE_ROOT = bootstrap_course_notebook(DEFAULT_COURSE_ROOT, mount_drive=False)


## Roboflow YOLO Detection Export 格式
Roboflow 匯出 YOLO 格式後，請將整個 dataset 放在：

```text
assets/datasets/roboflow_bbox_yolo/
├── data.yaml
├── train/ 或 images/train + labels/train
├── valid/ 或 images/valid + labels/valid
└── test/  或 images/test  + labels/test
```

本課程使用的 detector 類別與參考專案一致：`ball`、`ball-in-basket`、`number`、`player`、`player-in-possession`、`player-jump-shot`、`player-layup-dunk`、`player-shot-block`、`referee`、`rim`。

若學生是在 Roboflow 網頁上補標註，請先手動發布新的 dataset version；只有新 version 才能被下面的 API 下載流程抓到。


In [ ]:
from getpass import getpass
from src.roboflow_utils import DETECTION_CLASSES, dataset_status, ensure_roboflow_detection_dataset

BBOX_DATASET_DIR = COURSE_ROOT / "assets" / "datasets" / "roboflow_bbox_yolo"
DETECTOR_MODEL_PATH = (
    COURSE_ROOT / "assets" / "models" / "detectors" / "yolo26n_basketball_player_best.pt"
)

# 學生作業建議流程：
# 1. 在 Roboflow 網頁完成 bbox 標註。
# 2. 到 Versions 頁面按 Generate New Version / Publish。
# 3. 把新的 workspace / project / version 填在下面。
# 4. 將 USE_ROBOFLOW_DOWNLOAD 改成 True。
USE_ROBOFLOW_DOWNLOAD = False
ROBOFLOW_WORKSPACE = "YOUR_WORKSPACE"
ROBOFLOW_PROJECT = "YOUR_PROJECT"
ROBOFLOW_VERSION = 1
ROBOFLOW_API_KEY = ""
FORCE_DOWNLOAD = False

if USE_ROBOFLOW_DOWNLOAD:
    api_key = ROBOFLOW_API_KEY or getpass("Roboflow API key: ")
    data_yaml = ensure_roboflow_detection_dataset(
        BBOX_DATASET_DIR,
        workspace=ROBOFLOW_WORKSPACE,
        project=ROBOFLOW_PROJECT,
        version=ROBOFLOW_VERSION,
        api_key=api_key,
        export_format="yolov8",
        overwrite=FORCE_DOWNLOAD,
    )
    print("ready data.yaml:", data_yaml)

for i, name in enumerate(DETECTION_CLASSES):
    print(i, name)

print("\\ndataset status:")
print(dataset_status(BBOX_DATASET_DIR))
print("\\nmodel exists:", DETECTOR_MODEL_PATH.exists(), DETECTOR_MODEL_PATH)


In [ ]:
RUN_TRAINING = False

if RUN_TRAINING:
    from ultralytics import YOLO

    data_yaml = BBOX_DATASET_DIR / "data.yaml"
    if not data_yaml.exists():
        raise FileNotFoundError(
            "找不到 data.yaml。請先將 Roboflow YOLO detection export 放到 assets/datasets/roboflow_bbox_yolo/"
        )

    model = YOLO("yolo26n.pt")
    results = model.train(
        data=str(data_yaml),
        epochs=100,
        imgsz=960,
        batch=2,
        workers=4,
        patience=30,
        close_mosaic=10,
        project=str(COURSE_ROOT / "assets" / "results" / "training" / "bbox_detection"),
        name="yolo26n_basketball_player",
        plots=True,
    )
    print(results)
else:
    print("RUN_TRAINING=False；本課程預設使用 assets/models/detectors/ 內的已訓練權重。")


In [ ]:
import cv2
import pandas as pd
from ultralytics import YOLO

from src.cv_utils import save_image_rgb, save_json, show_image
from src.yolo_utils import detections_from_result

video_candidates = sorted((COURSE_ROOT / "assets" / "raw" / "reference_videos").glob("*.mp4"))
if not video_candidates:
    raise FileNotFoundError("找不到參考影片。請確認 assets/raw/reference_videos/ 內至少有一支 mp4。")

video_path = video_candidates[0]
cap = cv2.VideoCapture(str(video_path))
if not cap.isOpened():
    raise FileNotFoundError(video_path)
cap.set(cv2.CAP_PROP_POS_FRAMES, 15)
ok, frame_bgr = cap.read()
cap.release()
if not ok:
    raise RuntimeError(f"無法讀取 frame 15: {video_path}")
frame = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)

model = YOLO(str(DETECTOR_MODEL_PATH))
result = model.predict(frame, conf=0.25, imgsz=960, verbose=False)[0]
detections = detections_from_result(result, frame_index=15)
vis = result.plot()[:, :, ::-1].copy()
show_image(vis, "YOLO detections on reference frame")

out_image = COURSE_ROOT / "assets" / "results" / "d1_03_yolo_detections.png"
out_json = COURSE_ROOT / "assets" / "results" / "d1_03_yolo_detections.json"
save_image_rgb(out_image, vis)
save_json([record.__dict__ for record in detections], out_json)

print("video:", video_path)
print("detections:", len(detections))
print("saved:", out_image)
pd.DataFrame([record.__dict__ for record in detections]).head()
